Script analyse pour tous les participants

Chargement des bibliothèques

In [1]:
import os
import nibabel as nib
import pandas as pd
import numpy as np
from nilearn.input_data import NiftiLabelsMasker
from nilearn import datasets
from tqdm import tqdm  # progress bar

/tmp/ipykernel_701/2839967032.py:5: FutureWarning: The import path 'nilearn.input_data' is deprecated in version 0.9. Importing from 'nilearn.input_data' will be possible at least until release 0.13.0. Please import from 'nilearn.maskers' instead.
  from nilearn.input_data import NiftiLabelsMasker


Définition des chemins d'accès

In [9]:
base_path = "./ds003720" 
subject = ['sub-001', 'sub-002', 'sub-003', 'sub-004', 'sub-005']
n_runs = 6
t_r = 2.0  # repetition time (2 seconds)

Téléchargement de l'atlas et préparation du masque

In [3]:
# Load atlas & prepare masker
atlas = datasets.fetch_atlas_schaefer_2018(n_rois=100, resolution_mm=2)
masker = NiftiLabelsMasker(labels_img=atlas.maps, standardize=True, t_r=t_r)

# Prepare result lists
X_all = []
y_all = []

[fetch_atlas_schaefer_2018] Dataset found in /home/etudiants/nilearn_data/schaefer_2018

Vérification de la présence des fichiers et téléchargement des données

In [10]:
for subject in subject:
    print(f"\n Analyse du sujet {subject}")
    
    subject_X = []  # Features par sujet (pour matrices de confusion futures)
    subject_y = []
    
    # Loop through all 6 runs
    for run in tqdm(range(1, n_runs + 1), desc=f"{subject}"):
        run_str = f"run-0{run}"
        bold_file = f"{base_path}/{subject}/func/{subject}_task-Test_{run_str}_bold.nii"  
        events_file = f"{base_path}/{subject}/func/{subject}_task-Test_{run_str}_events.tsv"

        print(f"  Looking for: {bold_file}")
        print(f"  Looking for: {events_file}")

        if not os.path.exists(bold_file) or not os.path.exists(events_file):
            print(f"   Missing: {run_str}")
            continue

        # Load data
        func_img = nib.load(bold_file)
        events = pd.read_csv(events_file, sep="\t")  # \t sans \\
        roi_ts = masker.fit_transform(func_img)  # shape: [timepoints, 100]

        # Segment into trials
        for _, row in events.iterrows():
            onset = row['onset']
            duration = row['duration']
            genre = row['genre'].strip("'").strip('"')

            start_vol = int(onset / t_r)
            end_vol = int((onset + duration) / t_r)
            trial_ts = roi_ts[start_vol:end_vol, :]

            if trial_ts.shape[0] < 2:
                continue  # skip too-short segments

            # Compute connectivity
            conn_matrix = np.corrcoef(trial_ts.T)

            # Flatten upper triangle (4950 pour 100 ROIs)
            try:
                flat = conn_matrix[np.triu_indices_from(conn_matrix, k=1)]
                if not np.isnan(flat).any() and len(flat) == 4950:
                    subject_X.append(flat)  # Par sujet d'abord
                    subject_y.append(genre)
                    X_all.append(flat)      # Puis global
                    y_all.append(genre)
            except Exception as e:
                print(f"  ⚠️ Skipped a trial due to shape or NaN issue: {e}")
    
    # Sauvegarde par sujet (optionnel, pour debug/matrices de confusion)
    if subject_X:
        np.save(f"{subject}_X.npy", np.array(subject_X))
        np.save(f"{subject}_y.npy", np.array(subject_y))
        print(f"   {subject}: {len(subject_X)} trials valides")
    else:
        print(f"   {subject}: Aucun trial valide!")

print(f"\n Total: {len(X_all)} trials across {len(subject)} subjects")


 Analyse du sujet sub-001


sub-001:   0%|                                                                        | 0/6 [00:00<?, ?it/s]

  Looking for: ./ds003720/sub-001/func/sub-001_task-Test_run-01_bold.nii
  Looking for: ./ds003720/sub-001/func/sub-001_task-Test_run-01_events.tsv


/tmp/ipykernel_701/70597115.py:23: UserWarning: After resampling the label image to the data image, the following labels were removed: {np.float32(4.0), np.float32(54.0)}. Label image only contains 99 labels (including background).
  roi_ts = masker.fit_transform(func_img)  # shape: [timepoints, 100]
/tmp/ipykernel_701/70597115.py:23: FutureWarning: The 'zscore' strategy incorrectly uses population std to calculate sample zscores. The new strategy 'zscore_sample' corrects this behavior by using the sample std. In release 0.14.0, the 'zscore' option will be removed and using standardize=True will fall back to 'zscore_sample'.To avoid this warning, please use 'zscore_sample' instead.
  roi_ts = masker.fit_transform(func_img)  # shape: [timepoints, 100]
sub-001:  17%|██████████▋                                                     | 1/6 [00:18<01:32, 18.55s/it]

  Looking for: ./ds003720/sub-001/func/sub-001_task-Test_run-02_bold.nii
  Looking for: ./ds003720/sub-001/func/sub-001_task-Test_run-02_events.tsv


...

  Looking for: ./ds003720/sub-005/func/sub-005_task-Test_run-05_bold.nii
  Looking for: ./ds003720/sub-005/func/sub-005_task-Test_run-05_events.tsv


/tmp/ipykernel_701/70597115.py:23: FutureWarning: The 'zscore' strategy incorrectly uses population std to calculate sample zscores. The new strategy 'zscore_sample' corrects this behavior by using the sample std. In release 0.14.0, the 'zscore' option will be removed and using standardize=True will fall back to 'zscore_sample'.To avoid this warning, please use 'zscore_sample' instead.
  roi_ts = masker.fit_transform(func_img)  # shape: [timepoints, 100]
sub-005:  83%|█████████████████████████████████████████████████████▎          | 5/6 [01:24<00:16, 16.77s/it]

  Looking for: ./ds003720/sub-005/func/sub-005_task-Test_run-06_bold.nii
  Looking for: ./ds003720/sub-005/func/sub-005_task-Test_run-06_events.tsv


/tmp/ipykernel_701/70597115.py:23: FutureWarning: The 'zscore' strategy incorrectly uses population std to calculate sample zscores. The new strategy 'zscore_sample' corrects this behavior by using the sample std. In release 0.14.0, the 'zscore' option will be removed and using standardize=True will fall back to 'zscore_sample'.To avoid this warning, please use 'zscore_sample' instead.
  roi_ts = masker.fit_transform(func_img)  # shape: [timepoints, 100]
sub-005: 100%|████████████████████████████████████████████████████████████████| 6/6 [01:41<00:00, 16.85s/it]

   sub-005: 164 trials valides

 Total: 738 trials across 7 subjects


Convertir en liste numpy et sauvegarder des fichier pour l'utiliser dans le script de visualisation

In [11]:
# Convert to numpy arrays
X_all = np.array(X_all)
y_all = np.array(y_all)

print("✅ All runs processed!")
print("Total trials:", X_all.shape[0])
print("Feature shape per trial:", X_all.shape[1])
print("Unique genres:", np.unique(y_all))

# 💾 Save X_all for later visualization use
np.save("X_all.npy", X_all)

✅ All runs processed!
Total trials: 738
Feature shape per trial: 4950
Unique genres: ['blues' 'classical' 'country' 'disco' 'hiphop' 'jazz' 'metal' 'pop'
 'reggae' 'rock']


In [13]:
for subject in subject:
    if os.path.exists(f"{subject}_X.npy"):
        X_s = np.load(f"{subject}_X.npy")
        # Reconstruis matrices (100x100)
        conn_means = np.zeros((100, 100))
        for flat in X_s[:50]:  # Top 50 trials
            mat = np.zeros((100, 100))
            mat[np.triu_indices_from(mat, k=1)] = flat
            mat += mat.T  # Symétrise
            np.fill_diagonal(mat, 1)
            conn_means += mat / 2
        conn_means /= 50
        plotting.plot_matrix(conn_means, title=f"{subject} mean conn")  # nilearn